# Phase 3 – Complete Engine Verification

This notebook proves that the real online ASR retrieval engine can load the Phase 2 searchable corpus package, keep E5 and BGE resident together on a Colab T4, and execute both first-stage and reranked search.

This is a production smoke test. It does not measure retrieval accuracy and it does not rerun Parakeet.

## Optional dependency install

**What we are doing:** Install the pinned Transformers and SentenceTransformers versions used by the production retrieval environment.

**Why we are doing it:** Phase 3 needs the real E5 query encoder and BGE reranker. NeMo is intentionally not installed because online retrieval must not depend on the ASR runtime.

**PASS means:** Installation finishes without dependency errors. On a fresh Colab runtime, restart the session once after this cell if Colab asks you to.

In [1]:
import subprocess
import sys

command = [
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "transformers==4.52.0",
    "sentence-transformers==5.4.0",
    "pytest",
]

print("Running:", " ".join(command))
subprocess.run(command, check = True)

Running: /usr/bin/python3 -m pip install -q transformers==4.52.0 sentence-transformers==5.4.0 pytest


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', 'transformers==4.52.0', 'sentence-transformers==5.4.0', 'pytest'], returncode=0)

# 1. Environment and production tests

## Load the project and test the copied production code

**What we are doing:** Mount Drive, point Python at `production/`, confirm a plain package import does not pull in NeMo, Torch, or SentenceTransformers, then run every current production test.

**Why we are doing it:** This separates code-copy problems from real GPU/model problems. The package must also stay lightweight until the online engine explicitly loads E5 and BGE.

**PASS means:** CUDA sees a T4, the lightweight import check succeeds, and the current `tests/production/` suite exits successfully. We do not hard-code the number of tests because it may grow.

In [2]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os
import subprocess
import sys

PROJECT_ROOT = Path(
    "/content/drive/.shortcut-targets-by-id/"
    "1VXCKOD57uox_lUy-66Vqz5cKfzk8cvs4/aic26/asr_model_comparison"
)
PRODUCTION_ROOT = PROJECT_ROOT / "production"

assert PROJECT_ROOT.exists(), PROJECT_ROOT
assert (PRODUCTION_ROOT / "asr_retrieval").exists(), PRODUCTION_ROOT

os.chdir(PROJECT_ROOT)

if (str(PRODUCTION_ROOT) not in sys.path) :
    sys.path.insert(0, str(PRODUCTION_ROOT))

import_check = f'''
import sys
sys.path.insert(0, {str(PRODUCTION_ROOT)!r})
import asr_retrieval
assert "nemo" not in sys.modules
assert "torch" not in sys.modules
assert "sentence_transformers" not in sys.modules
print("Lightweight import: PASS")
'''

subprocess.run(
    [sys.executable, "-c", import_check],
    check = True,
)

import torch
import transformers
import sentence_transformers

print("Project:", PROJECT_ROOT)
print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("SentenceTransformers:", sentence_transformers.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

assert torch.cuda.is_available()
assert "T4" in torch.cuda.get_device_name(0)

print("\\nRunning current production tests...")
subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "tests/production"],
    cwd = PROJECT_ROOT,
    check = True,
)
print("Production test suite: PASS")

Mounted at /content/drive
Project: /content/drive/.shortcut-targets-by-id/1VXCKOD57uox_lUy-66Vqz5cKfzk8cvs4/aic26/asr_model_comparison
Torch: 2.11.0+cu128
Transformers: 4.52.0
SentenceTransformers: 5.4.0
CUDA available: True
GPU: Tesla T4
\nRunning current production tests...
Production test suite: PASS


# 2. Load the Phase 2 searchable corpus package

## Confirm the saved text, BM25, E5, and mappings still belong together

**What we are doing:** Find the Phase 2 smoke release, run the production release validator, then load the saved corpus package.

**Why we are doing it:** `all transcript windows` means every recorded overlapping transcript window. `searchable transcript windows` means only the windows with usable retrieval text. BM25 and the saved E5 matrix must both contain exactly the searchable-window count, and each E5 row must still map to the same transcript window it was built from.

**PASS means:** File hashes are valid, the saved mappings are internally consistent, BM25 document count equals the searchable-window count, and E5 has one 1024-dimensional row for every searchable window.

In [3]:
from asr_retrieval.artifacts import (
    load_release,
    resolve_active_release,
    validate_release,
)

RELEASE_PATH_OVERRIDE = None
RELEASES_ROOT = PROJECT_ROOT / "production_smoke" / "phase2" / "releases"

if (RELEASE_PATH_OVERRIDE is not None) :
    RELEASE_PATH = Path(RELEASE_PATH_OVERRIDE)
elif ((RELEASES_ROOT / "CURRENT").exists()) :
    RELEASE_PATH = resolve_active_release(RELEASES_ROOT)
else :
    candidates = sorted(
        [
            path
            for path in RELEASES_ROOT.iterdir()
            if path.is_dir() and ".building." not in path.name
        ],
        key = lambda path : path.stat().st_mtime,
        reverse = True,
    )
    RELEASE_PATH = None
    for candidate in candidates :
        try :
            validate_release(candidate, verify_hashes = True)
            RELEASE_PATH = candidate
            break
        except (FileNotFoundError, ValueError) as error :
            print("Skipping invalid release:", candidate.name, "-", error)

    if (RELEASE_PATH is None) :
        raise FileNotFoundError(
            "No valid Phase 2 release was found. "
            "Set RELEASE_PATH_OVERRIDE to the release directory."
        )

manifest = validate_release(RELEASE_PATH, verify_hashes = True)
release = load_release(RELEASE_PATH)

assert release.bm25.document_count == manifest.eligible_window_count
assert release.embeddings.shape == (manifest.eligible_window_count, 1024)

expected_ids = tuple(
    release.windows[int(physical_index)].window_id
    for physical_index in release.eligible_to_physical
)
assert expected_ids == release.eligible_window_ids

print("Release:", RELEASE_PATH)
print("Release ID:", manifest.release_id)
print("Videos:", manifest.video_count)
print("All transcript windows:", manifest.physical_window_count)
print("Searchable transcript windows:", manifest.eligible_window_count)
print("E5 matrix:", release.embeddings.shape, release.embeddings.dtype)
print("BM25 documents:", release.bm25.document_count)
print("Saved corpus package: PASS")

Release: /content/drive/.shortcut-targets-by-id/1VXCKOD57uox_lUy-66Vqz5cKfzk8cvs4/aic26/asr_model_comparison/production_smoke/phase2/releases/phase2-smoke-002
Release ID: phase2-smoke-002
Videos: 1
All transcript windows: 29
Searchable transcript windows: 29
E5 matrix: (29, 1024) float32
BM25 documents: 29
Saved corpus package: PASS


# 3. Load E5 and BGE together

## Prove the complete engine fits in one T4 process

**What we are doing:** Load the normal `ASRRetrievalEngine` from the saved corpus package. The public loader loads E5 and BGE together, so we measure GPU memory before loading, after both models are resident, and after the normal engine warmup.

**Why we are doing it:** The final online service must keep both neural models loaded. It must not unload one model and reload another for every query.

**PASS means:** Engine state reaches `READY`, both models report loaded, warmup succeeds, and CUDA does not run out of memory.

In [4]:
from asr_retrieval import ASRRetrievalEngine, ProductionConfig

def gpu_memory(label) :
    torch.cuda.synchronize()
    return {
        "label" : label,
        "allocated_gib" : torch.cuda.memory_allocated() / (1024 ** 3),
        "reserved_gib" : torch.cuda.memory_reserved() / (1024 ** 3),
    }

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

memory_before = gpu_memory("before engine load")

production_config = ProductionConfig()
engine = ASRRetrievalEngine.from_artifacts(
    RELEASE_PATH,
    config = production_config,
    warmup = False,
)

memory_loaded = gpu_memory("E5 + BGE loaded")
warmup_timings = engine.warmup()
memory_warm = gpu_memory("after warmup")

assert engine.state == "READY"
assert engine.e5.loaded
assert engine.reranker is not None and engine.reranker.loaded

for record in [memory_before, memory_loaded, memory_warm] :
    print(
        f"{record['label']:<20} "
        f"allocated={record['allocated_gib']:.3f} GiB  "
        f"reserved={record['reserved_gib']:.3f} GiB"
    )

print("Warmup:", warmup_timings)
print("Combined E5 + BGE load: PASS")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/128 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_xlm-roberta_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

before engine load   allocated=0.000 GiB  reserved=0.000 GiB
E5 + BGE loaded      allocated=3.143 GiB  reserved=3.160 GiB
after warmup         allocated=3.152 GiB  reserved=3.166 GiB
Warmup: {'e5_encode_ms': 719.5725349999975, 'dense_search_ms': 0.16861299991433043, 'bge_ms': 186.25859200005834}
Combined E5 + BGE load: PASS


# 4. Run first-stage search without BGE

## Confirm BM25 + E5 produces a deterministic video ranking

**What we are doing:** Build one smoke query from text that really exists in the saved corpus, then run first-stage-only search twice.

**Why we are doing it:** This isolates the path `query → BM25 + E5 → separate normalization → 25% BM25 + 75% E5 → window scores → best window per video`. BGE must not be called in this mode.

**PASS means:** Scores are finite, ranks are contiguous, each supporting window belongs to its stated video, BGE pair count is zero, and repeating the same query returns the same video order.

In [5]:
import numpy as np

searchable_physical = release.eligible_to_physical.astype(int).tolist()
query_source_index = max(
    searchable_physical,
    key = lambda index : len(release.windows[index].retrieval_text.split()),
)
query_source_text = release.windows[query_source_index].retrieval_text
QUERY = " ".join(query_source_text.split()[:16])

SEARCH_TOP_K = manifest.video_count

first_stage_1 = engine.search(
    QUERY,
    top_k = SEARCH_TOP_K,
    first_stage_only = True,
    windows_per_hit = 2,
)
first_stage_2 = engine.search(
    QUERY,
    top_k = SEARCH_TOP_K,
    first_stage_only = True,
    windows_per_hit = 2,
)

assert first_stage_1.mode == "first_stage"
assert first_stage_1.timings.candidate_pair_count == 0
assert [hit.video_id for hit in first_stage_1.hits] == [
    hit.video_id for hit in first_stage_2.hits
]
assert [hit.rank for hit in first_stage_1.hits] == list(
    range(1, len(first_stage_1.hits) + 1)
)

for hit in first_stage_1.hits :
    assert np.isfinite(hit.first_stage_score)
    assert hit.final_score is None
    assert hit.reranker_score is None
    for window in hit.windows :
        assert window.video_id == hit.video_id
        assert np.isfinite(window.first_stage_score)

print("Query:", QUERY)
print("\nTop first-stage results:")
for hit in first_stage_1.hits[:5] :
    window = hit.windows[0] if hit.windows else None
    interval = (
        f"{window.start_s:.1f}–{window.end_s:.1f}s"
        if window is not None
        else "no searchable transcript"
    )
    excerpt = (
        window.transcript[:140].replace("\n", " ")
        if window is not None
        else ""
    )
    print(
        f"{hit.rank:>2}. {hit.video_id:<18} "
        f"score={hit.first_stage_score:.4f}  "
        f"{interval}  {excerpt}"
    )

print("\nRepeated first-stage order: identical")
print("BGE pairs:", first_stage_1.timings.candidate_pair_count)
print("First-stage search: PASS")

Query: ch và khám xét chỗ ở đối với lê thị kim quyên sinh năm một ngàn

Top first-stage results:
 1. L21_V015           score=1.0000  540.0–600.0s  ch và khám xét chỗ ở đối với lê thị kim quyên sinh năm một ngàn chín trăm tám mươi lăm trú tỉnh tây ninh mỹ can là giám đốc công ty trách nh

Repeated first-stage order: identical
BGE pairs: 0
First-stage search: PASS


# 5. Run the complete BGE-reranked search

## Confirm the 150-pair cap and final ordering rules

**What we are doing:** Run the same query through the normal reranked search twice and inspect the actual result fields and timing fields.

**Why we are doing it:** BGE may reorder only the candidate-video prefix. Videos that were not reranked must remain in their original BM25 + E5 relative order. Candidate final scores are normalized inside the candidate set, so they are never numerically mixed with noncandidate first-stage scores.

**PASS means:** Pair count stays within the configured cap, reranked videos form a prefix, any noncandidate suffix keeps increasing first-stage ranks, reranked supporting windows come first for candidate videos, and repeating the query gives the same final video order.

In [6]:
def first_stage_latency_ms(timings) :
    return (
        timings.bm25_ms
        + timings.e5_encode_ms
        + timings.dense_search_ms
        + timings.first_stage_fusion_ms
        + timings.video_aggregation_ms
    )

full_1 = engine.search(
    QUERY,
    top_k = SEARCH_TOP_K,
    windows_per_hit = 2,
)
full_2 = engine.search(
    QUERY,
    top_k = SEARCH_TOP_K,
    windows_per_hit = 2,
)

assert full_1.mode == "reranked"
assert full_1.timings.candidate_pair_count <= production_config.candidates.max_candidate_pairs
assert [hit.video_id for hit in full_1.hits] == [hit.video_id for hit in full_2.hits]

flags = [hit.reranked for hit in full_1.hits]
seen_noncandidate = False
for reranked in flags :
    if (not reranked) :
        seen_noncandidate = True
    elif (seen_noncandidate) :
        raise AssertionError("A reranked candidate appeared after the noncandidate suffix")

noncandidate_first_stage_ranks = [
    hit.first_stage_rank
    for hit in full_1.hits
    if not hit.reranked
]
assert noncandidate_first_stage_ranks == sorted(noncandidate_first_stage_ranks)

for hit in full_1.hits :
    if (hit.reranked) :
        assert hit.final_score is not None
        assert hit.windows
        assert hit.windows[0].reranked
    else :
        assert hit.final_score is None
        assert hit.reranker_score is None
    for window in hit.windows :
        assert window.video_id == hit.video_id
        assert np.isfinite(window.first_stage_score)
        if (window.reranker_score is not None) :
            assert np.isfinite(window.reranker_score)
        if (window.final_score is not None) :
            assert np.isfinite(window.final_score)

candidate_video_count = sum(hit.reranked for hit in full_1.hits)

print("Top reranked results:")
for hit in full_1.hits[:5] :
    window = hit.windows[0] if hit.windows else None
    interval = (
        f"{window.start_s:.1f}–{window.end_s:.1f}s"
        if window is not None
        else "no searchable transcript"
    )
    final_text = f"{hit.final_score:.4f}" if hit.final_score is not None else "n/a"
    excerpt = (
        window.transcript[:130].replace("\n", " ")
        if window is not None
        else ""
    )
    print(
        f"{hit.rank:>2}. {hit.video_id:<18} "
        f"first_rank={hit.first_stage_rank:<4} "
        f"reranked={str(hit.reranked):<5} "
        f"final={final_text:<8} {interval}  {excerpt}"
    )

print("\nCandidate videos:", candidate_video_count)
print("BGE pairs:", full_1.timings.candidate_pair_count)
print("BGE batch size:", full_1.timings.reranker_batch_size)
print("First-stage latency:", f"{first_stage_latency_ms(full_1.timings):.2f} ms")
print("BGE latency:", f"{full_1.timings.bge_ms:.2f} ms")
print("Total latency:", f"{full_1.timings.total_ms:.2f} ms")
print("Final repeated order: identical")

if (noncandidate_first_stage_ranks) :
    print("Noncandidate suffix order: PASS")
else :
    print("Noncandidate suffix order: N/A – every video in this small corpus was reranked")

print("Full reranked search: PASS")

Top reranked results:
 1. L21_V015           first_rank=1    reranked=True  final=1.0000   540.0–600.0s  ch và khám xét chỗ ở đối với lê thị kim quyên sinh năm một ngàn chín trăm tám mươi lăm trú tỉnh tây ninh mỹ can là giám đốc công t

Candidate videos: 1
BGE pairs: 3
BGE batch size: 32
First-stage latency: 34.82 ms
BGE latency: 165.49 ms
Total latency: 200.87 ms
Final repeated order: identical
Noncandidate suffix order: N/A – every video in this small corpus was reranked
Full reranked search: PASS


# 6. Phase 3 verdict

## Summarize only the checks actually performed

**What we are doing:** Print the final Phase 3 smoke-test result and the main measured runtime values.

**Why we are doing it:** This gives one clear answer to whether the real Phase 3 engine works on the T4 before we start the broader Phase 4 production checks.

**PASS means:** Production tests passed, the saved corpus loaded correctly, E5 and BGE stayed resident together, first-stage search worked without BGE, full reranked search respected the pair cap, and repeated searches produced identical video ordering.

In [7]:
phase3_checks = {
    "Production tests" : True,
    "Phase 2 corpus load" : True,
    "E5 + BGE simultaneous T4 load" : True,
    "First-stage search" : True,
    "Full reranked search" : True,
    "Candidate pair cap" : (
        full_1.timings.candidate_pair_count
        <= production_config.candidates.max_candidate_pairs
    ),
    "Deterministic repeat" : (
        [hit.video_id for hit in full_1.hits]
        == [hit.video_id for hit in full_2.hits]
    ),
}

print("PHASE 3 VERIFICATION\n")
for name, passed in phase3_checks.items() :
    print(f"{name}: {'PASS' if passed else 'FAIL'}")

print("\nMeasured values")
print(
    "GPU after warmup:",
    f"{memory_warm['allocated_gib']:.3f} GiB allocated,",
    f"{memory_warm['reserved_gib']:.3f} GiB reserved",
)
print(
    "First-stage query latency:",
    f"{first_stage_latency_ms(full_1.timings):.2f} ms",
)
print("Full query latency:", f"{full_1.timings.total_ms:.2f} ms")
print("BGE pair count:", full_1.timings.candidate_pair_count)

phase3_pass = all(phase3_checks.values())
print("\nPhase 3 status:", "PASS" if phase3_pass else "FAIL")

if (not phase3_pass) :
    raise AssertionError("Phase 3 verification failed")

engine.close()

PHASE 3 VERIFICATION

Production tests: PASS
Phase 2 corpus load: PASS
E5 + BGE simultaneous T4 load: PASS
First-stage search: PASS
Full reranked search: PASS
Candidate pair cap: PASS
Deterministic repeat: PASS

Measured values
GPU after warmup: 3.152 GiB allocated, 3.166 GiB reserved
First-stage query latency: 34.82 ms
Full query latency: 200.87 ms
BGE pair count: 3

Phase 3 status: PASS
